# Bölüm 7 · Varyans Analizi (ANOVA)

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b07/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b07/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b07/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b07/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b07/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. Omnibus ANOVA ile tekil karşıtlıkları ayırır.
2. Etkileşimi koşullu grup farklarıyla yorumlar.
3. Karşılaştırma ailesini ve belirsizliği açık raporlar.

**İleri rota:** Dengesiz hedefler ve HC3 karşıtlıkları; çoklu doğrusal model fikri için Bölüm 13 doğrusal kısmına başvurulabilir.

**Ç1 · Çıkış sorusu:** Üç gruplu ANOVA anlamlıysa bütün grup çiftleri farklı mıdır?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b07/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b07-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b07-varyans-analizi'
modules=['uygulamalar/b07-varyans-analizi/dengesiz-hedef']
expected_main=json.loads((root/"bolumler/b07/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 7 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b07/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 7 — görevler

## D1–D3: Analizden Önce

1.  İki faktör, düzeyleri, hücre sayısı ve analiz birimi nedir? Bu veri neden tekrarlı ölçümler ANOVA’sı değildir?

2.  OJ içindeki doz sorusu ile bütün kayıtlardaki uygulama–doz etkileşimi hangi farklı hipotezleri sınar?

3.  Dozu sayısal bir eğim yerine üç düzeyli faktör olarak yazmak hangi kısıtı kaldırır? 0.5, 1 ve 2’nin eşit aralıklı olmadığını dikkate alın.

Etkinlik yanıtları Ek F’de, Bölüm 7 başlığı altındadır (s. kitaptaki ilgili sayfa).

# Rehberli ve Bağımsız Etkinlikler

## G1–G3: Rehberli Hesaplar

1.  OJ ortalamaları ve standart sapmalarından $SSG$, $SSW$, $SST$, F, $\eta^2$ ve $\omega^2$ değerlerini yeniden hesaplayın. Neden hata serbestlik derecesi 29 değil 27’dir?

2.  OJ grubunda 2$-$<!-- -->1 farkının Tukey aralığını kurun. Genel F anlamlıyken bu aralığın sıfırı içermesi çelişki midir? Games–Howell ile yön ve belirsizliği karşılaştırın.

3.  Tam modelde dört KT bileşenini toplayın, etkileşim F’sini ve kısmi eta-kareyi bulun. 1 mg/gün OJ$-$VC aralığını $MS_E=13.18715$, $sd=54$ ve üçlü Bonferroni düzeltmesiyle üretin.

## B1–B3: VC Grubunda Bağımsız Çözüm

Yalnız VC alan 30 kaydı seçin; OJ hata terimini veya katsayılarını taşımayın.

1.  Üç dozun özetlerini, klasik ANOVA tablosunu, eta-kare ve omega-kareyi hesaplayın.

2.  Üç çiftlik Tukey ailesini kurun; ardından Welch/Games–Howell ile sonuçları karşılaştırın. Hangi çiftlerin aralıkları sıfırı içerir?

3.  En fazla 100 sözcüklük bir raporda örneklem kapsamını, fark yönünü, karşılaştırma ailesini ve eş varyans varsayımını belirtin. OJ ile VC’nin ayrı testlerinden etkileşim sonucuna atlamayın.

## H1: Altı Hatalı İddiayı Düzeltin

Her iddiayı bu bölümdeki bir formül veya sayısal sonuçla düzeltin.

## P1: Ortak Model ve Çoklu Karşılaştırma Denetimi

60 kayıt ve altı eşit hücreyi doğrulayın. OJ klasik/Welch sonuçlarını, Tukey/Games–Howell ailelerini, faktöriyel KT ayrıştırmasını ve üç basit etkinin Bonferroni aralıklarını yeniden üretin. Klasik etkileşimle HC3 sonucunu karşılaştırın. Teslim: çalışan kod, üç grafik, kaynak/analiz sözlüğü ve bir sayfalık eleştirel rapor.

<div class="center">

| Ölçüt                                              | Puan |
|:---------------------------------------------------|-----:|
| Kaynak, hücreler ve faktör kodlaması               |   20 |
| Kareler toplamları ve etki büyüklükleri            |   25 |
| Karşılaştırma ailesi ve aralıkların eşleştirilmesi |   25 |
| Etkileşim, tanılar ve HC3 duyarlılığı              |   20 |
| Yeniden üretim ve sınırlılık raporu                |   10 |

</div>

## T1–T3: Hedef ve Etkileşim Denetimi

1.  Öğretim alt kümesindeki 4.376111, 4.220660 ve 5.654783 farklarının ağırlıklarını açıklayın. Hangisi grupların ayrı doz bileşimini de taşır?

2.  Bir dozda p küçük, diğerinde büyük bulunması neden etkileşim testi değildir? İki planlı farkların farkının ailesini ve HC3 t yaklaşımının sınırını belirtin.

3.  Kaynak sırasıyla 48 kayıt seçiminin ana 60 kaydı veya rastgele kayıp varsayımını neden değiştirmediğini açıklayın. Tip III etiketi hedef kontrastı açıklamaya tek başına yeter mi?

- Üç veya daha fazla grupta karşılaştırma ailesini tanımlamadan düzeltmesiz t-testleri taramak.

- Genel F testini bütün ikili testleri düzeltmesiz yapma izni saymak veya her karşıtlık için zorunlu anlamlılık kapısı sanmak.

- Varyans homojenliği varsayımını kontrol etmemek.

- ANOVA’nın hangi grupların farklı olduğunu doğrudan söylediğini sanmak.

- Etki büyüklüğünü raporlamadan yalnızca $p$-değerine odaklanmak.

- Etkileşim anlamlıyken yalnızca ana etkileri yorumlamak.

Bu hataları önlemek için ANOVA sonuçları her zaman varsayımlar, genel test, etki büyüklüğü ve post-hoc aşamalarıyla birlikte ele alınmalıdır.

# Alıştırmalar

## Kolay Düzey

1.  Tek yönlü ANOVA’nın sıfır hipotezi nedir?

2.  ANOVA’da gruplar içi değişkenliği hangi kareler toplamı gösterir?

3.  Genel ANOVA sonucu anlamlıysa sonraki adım genellikle nedir?

4.  Parametrik koşullar zayıfsa tek yönlü ANOVA’ya alternatif ne olabilir?

5.  $\eta^2$ değeri neyi gösterir?

## Orta Düzey

1.  Dört öğretim yönteminin başarı puanları karşılaştırılacaktır. ANOVA neden çoklu t-testlerine tercih edilmelidir?

2.  ANOVA sonucunda $p<.05$ bulunmuştur. Bu sonuç neyi gösterir, neyi göstermez?

3.  İki yönlü ANOVA’da etkileşim etkisi anlamlı bulunmuştur. Nasıl yorumlanır?

4.  Tek yönlü ANOVA’da $F(3,76)=4.92$, $p=.004$ raporlanmıştır. Kaç grup vardır ve genel sonuç nedir?

5.  Varyans homojenliği ihlal edilmiş görünmektedir. Araştırmacı rapordan önce neyi değerlendirmelidir?

## İleri Düzey

1.  ANOVA anlamlı, Tukey karşılaştırmalarında yalnızca bir çift anlamlıdır. Nasıl raporlanır?

2.  İki yönlü ANOVA’da ana etkiler ve etkileşim anlamlıdır. Ana etkileri tek başına yorumlamak neden sakıncalıdır?

3.  Çok büyük örneklemde $p<.001$, $\eta^2=0.02$ bulunmuştur. İstatistiksel ve pratik önem nasıl dengelenir?

4.  Araştırmacı doğal üç grubu ANOVA ile karşılaştırıp grup üyeliğinin sonucu “nedenlediğini” yazmıştır. Bu neden sorunludur?

5.  Dengesiz faktöriyel ANOVA’da hücre büyüklükleri çok farklıdır. Hangi ek konular dikkate alınmalıdır?


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b07/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b07/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Eşit doz ağırlığı veya ortak gözlenen doz ağırlığı hedefini seçin; tahminin değişmesini hedef farkıyla açıklayın.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 7,
    "gorev": 'Eşit doz ağırlığı veya ortak gözlenen doz ağırlığı hedefini seçin; tahminin değişmesini hedef farkıyla açıklayın.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":7,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b07-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b07-varyans-analizi',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)